# PQID — Instruction Generation Pipeline

Generates natural-language instructions for the 2,298 new circuits found during the dataset overhaul.

**Run cells in order:**
1. Setup (key + asyncio patch)
2. Stage 1 — Seed generation (one instruction per circuit)
3. Stage 2 — Paraphrase generation (5 variants per seed)
4. Stage 3 — Merge into main dataset

Each stage is resume-safe — re-running skips already completed entries.

In [1]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os, subprocess, sys

# Python interpreter that has openai installed
PYTHON = r'c:\Users\Abebe\torch311env\Scripts\python.exe'

# Load OpenAI API key from file and store for subprocess env
KEY_PATH = r'C:\Users\Abebe\Downloads\IT\OPENAI\OPENAI_API_KEY_PQID_V2.txt'
with open(KEY_PATH, 'r') as f:
    os.environ['OPENAI_API_KEY'] = f.read().strip()
print('API key loaded:', 'sk-...' + os.environ['OPENAI_API_KEY'][-6:])

# Helper: run a script with torch311env Python, streaming output in real-time
def run_script(script_path):
    env = {**os.environ}   # includes OPENAI_API_KEY
    proc = subprocess.Popen(
        [PYTHON, script_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'\nScript exited with code {proc.returncode}')
    else:
        print('\nDone.')

# Verify the interpreter works
result = subprocess.run([PYTHON, '-c', 'import openai; print("openai", openai.__version__, "ready")'],
                        capture_output=True, text=True, env=os.environ)
print(result.stdout.strip())
if result.stderr:
    print('Warning:', result.stderr.strip())

API key loaded: sk-...-g77EA


openai 1.77.0 ready


In [4]:
# ── Cell 2: Stage 1 — Seed generation ─────────────────────────────────────
# Generates one instruction per circuit (~15-20 min, ~2,298 API calls)
# Resume-safe: re-running skips already completed circuits

STAGE1 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
          r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
          r'\scripts\03_instruction_generation\generate_seeds_pending.py')

run_script(STAGE1)

Circuits to process: 2298
Already done: 2295  Remaining: 3
  3/3 | ok=3 err=0 | tokens=22216 | ETA 0:00:00

Done in 0:00:01
  Success: 3  Errors: 0  Tokens: 22216
  Est. cost: $0.013

Done.


In [5]:
# ── Cell 3: Check Stage 1 output before proceeding ─────────────────────────
import json

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

seeds = load_jsonl(f'{BASE}/seeds_pending.jsonl')
errors_path = f'{BASE}/seeds_pending_errors.jsonl'
errors = load_jsonl(errors_path) if os.path.exists(errors_path) else []

print(f'Seeds generated:  {len(seeds)} / 2298')
print(f'Errors logged:    {len(errors)}')
print()
print('Sample entries:')
for e in seeds[:3]:
    print(f'  INPUT:  {e["input"]}')
    print(f'  OUTPUT: {e["output"][:80]}...')
    print()

Seeds generated:  2298 / 2298
Errors logged:    3

Sample entries:
  INPUT:  Implement a quantum circuit with two qubits that performs a SWAP operation, exchanging the quantum states of the two qubits.
  OUTPUT: qc = QuantumCircuit(2)
# swaps states of qubits a and b
qc.swap(a,b)
qc.draw()...

  INPUT:  Create a quantum circuit with two qubits and apply no gate operations to it.
  OUTPUT: from qiskit import QuantumCircuit
from qiskit.circuit import Gate
from math impo...

  INPUT:  Implement a quantum circuit that performs a swap operation, moving a quantum state from qubit 'b' to 'a' using two CNOT gates.
  OUTPUT: # swap a q from b to a
qc.cx(b,a) # copies 1 from b to a
qc.cx(a,b) # uses the 1...



In [8]:
# ── Cell 4: Stage 2 — Paraphrase generation ────────────────────────────────
# Generates 5 paraphrases per seed (~20-25 min, ~2,298 API calls)
# Resume-safe: re-running skips already completed circuits

STAGE2 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
          r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
          r'\scripts\03_instruction_generation\generate_paraphrases_pending.py')

run_script(STAGE2)

Seeds loaded: 2298
Already done: 2297  Remaining: 1
  1/1 | ok=1 err=0 | tokens=366 | ETA 0:00:00

Done in 0:00:09
  Success: 1  Errors: 0
  Entries written: 5
  Tokens used: 366
  Est. cost: $0.000

Done.


In [ ]:
# ── Cell 5: Check Stage 2 output before merging ────────────────────────────
paras = load_jsonl(f'{BASE}/paraphrases_pending.jsonl')
para_errors_path = f'{BASE}/paraphrases_pending_errors.jsonl'
para_errors = load_jsonl(para_errors_path) if os.path.exists(para_errors_path) else []

unique_circuits = len({e['metadata']['circuit_hash'] for e in paras})
print(f'Paraphrases generated: {len(paras)}')
print(f'Unique circuits:       {unique_circuits} / 2298')
print(f'Avg per circuit:       {len(paras)/unique_circuits:.1f}' if unique_circuits else '')
print(f'Errors logged:         {len(para_errors)}')
print()
print('Sample paraphrase group:')
first_hash = paras[0]['metadata']['circuit_hash']
group = [e for e in paras if e['metadata']['circuit_hash'] == first_hash]
print(f'  Original: {group[0]["metadata"].get("original_prompt", "n/a")}')
for i, e in enumerate(group):
    print(f'  Para {i+1}:  {e["input"]}')

In [ ]:
# ── Cell 6: Stage 3 — Merge into main dataset ──────────────────────────────
# Merges seeds + paraphrases into train_clean.jsonl / validation_clean.jsonl
# Safe to re-run: deduplicates by input text before writing

STAGE3 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
          r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
          r'\scripts\03_instruction_generation\merge_new_entries.py')

run_script(STAGE3)

In [ ]:
# ── Cell 7: Final dataset summary (after Batch 1) ──────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_b1 = load_jsonl(f'{BASE}/train_clean.jsonl')
val_b1   = load_jsonl(f'{BASE}/validation_clean.jsonl')

total = len(train_b1) + len(val_b1)
flag_counts = {}
for e in train_b1 + val_b1:
    fl = e['metadata'].get('quality_flag', 'unknown')
    flag_counts[fl] = flag_counts.get(fl, 0) + 1

print('=== FINAL DATASET SUMMARY (after Batch 1) ===')
print(f'Train:  {len(train_b1)}')
print(f'Val:    {len(val_b1)}')
print(f'Total:  {total}')
print()
print('By quality_flag:')
for fl, c in sorted(flag_counts.items()):
    print(f'  {fl}: {c}')

In [ ]:
# ── Cell 8: Metadata enrichment ────────────────────────────────────────────
# Enriches train_clean.jsonl + validation_clean.jsonl with circuit-level
# metadata: num_qubits, circuit_depth, gate_count, gate_types, complexity_class, etc.
# Overwrites files in-place after both splits complete successfully.
# Runtime: ~10-20 min for 22K entries (local, no API calls)

ENRICH = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
          r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
          r'\scripts\enrich_metadata.py')

run_script(ENRICH)

In [ ]:
# ── Cell 9: GitHub expansion scraping ──────────────────────────────────────
# Searches GitHub for new Qiskit/OpenQASM repos not already in the dataset.
# Output: circuits_expansion.jsonl  (same schema as circuits_pending_instructions.jsonl)
# Resume-safe: re-running skips already-fetched URLs.
# Runtime: 30-90 min depending on GitHub rate limits.

import os

GITHUB_TOKEN_PATH = r'C:\Users\Abebe\Downloads\IT\GITHUB\GITHUB_TOKEN_PQID_V1.txt'
with open(GITHUB_TOKEN_PATH, 'r') as f:
    os.environ['GITHUB_TOKEN'] = f.read().strip()
print('GitHub token loaded:', 'ghp_...' + os.environ['GITHUB_TOKEN'][-6:])

SCRAPE_EXPANSION = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
                    r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
                    r'\scripts\scrape_github_expansion.py')

run_script(SCRAPE_EXPANSION)

## Batch 2 — GitHub Expansion Pipeline

Generates instructions for the 9,527 new circuits found via GitHub Search API.

**Run cells in order:**
- Cell 10 — Seed generation (one instruction per circuit)
- Cell 11 — Check seeds
- Cell 12 — Paraphrase generation (5 variants per seed)
- Cell 13 — Check paraphrases
- Cell 14 — Merge into main dataset
- Cell 15 — Final dataset summary

In [3]:
# ── Cell 10: Batch 2 Stage 1 — Seed generation ────────────────────────────
# Generates one instruction per circuit (~9,527 circuits)
# Resume-safe: re-running skips already completed circuits

STAGE1_EXP = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
              r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
              r'\scripts\03_instruction_generation\generate_seeds_expansion.py')

run_script(STAGE1_EXP)

Circuits to process: 9527
Already done: 1055  Remaining: 8472
  100/8472 | ok=100 err=0 | tokens=65271 | ETA 0:12:55
  200/8472 | ok=200 err=0 | tokens=116174 | ETA 0:09:23
  300/8472 | ok=300 err=0 | tokens=184602 | ETA 0:08:00
  400/8472 | ok=400 err=0 | tokens=235733 | ETA 0:07:19
  500/8472 | ok=500 err=0 | tokens=324012 | ETA 0:07:02
  600/8472 | ok=600 err=0 | tokens=389117 | ETA 0:06:56
  700/8472 | ok=700 err=0 | tokens=431315 | ETA 0:06:48
  800/8472 | ok=800 err=0 | tokens=482261 | ETA 0:06:41
  900/8472 | ok=900 err=0 | tokens=531817 | ETA 0:07:39
  1000/8472 | ok=1000 err=0 | tokens=581233 | ETA 0:07:17
  1100/8472 | ok=1100 err=0 | tokens=639076 | ETA 0:06:59
  1200/8472 | ok=1200 err=0 | tokens=691734 | ETA 0:06:44
  1300/8472 | ok=1300 err=0 | tokens=741961 | ETA 0:06:32
  1400/8472 | ok=1400 err=0 | tokens=786211 | ETA 0:06:21
  1500/8472 | ok=1500 err=0 | tokens=853390 | ETA 0:06:31
  1600/8472 | ok=1600 err=0 | tokens=941749 | ETA 0:06:44
  1700/8472 | ok=1700 err=0 |

In [ ]:
# ── Cell 11: Check Batch 2 Stage 1 output ─────────────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

seeds = load_jsonl(f'{BASE}/seeds_expansion.jsonl')
errors_path = f'{BASE}/seeds_expansion_errors.jsonl'
errors = load_jsonl(errors_path) if os.path.exists(errors_path) else []

print(f'Seeds generated:  {len(seeds)} / 9527')
print(f'Errors logged:    {len(errors)}')
print()
print('Sample entries:')
for e in seeds[:3]:
    print(f'  INPUT:  {e["input"]}')
    print(f'  OUTPUT: {e["output"][:80]}...')
    print()

In [ ]:
# ── Cell 12: Batch 2 Stage 2 — Paraphrase generation ──────────────────────
# Generates 5 paraphrases per seed (~9,527 circuits)
# Resume-safe: re-running skips already completed circuits

STAGE2_EXP = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
              r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
              r'\scripts\03_instruction_generation\generate_paraphrases_expansion.py')

run_script(STAGE2_EXP)

In [ ]:
# ── Cell 13: Check Batch 2 Stage 2 output ─────────────────────────────────
paras = load_jsonl(f'{BASE}/paraphrases_expansion.jsonl')
para_errors_path = f'{BASE}/paraphrases_expansion_errors.jsonl'
para_errors = load_jsonl(para_errors_path) if os.path.exists(para_errors_path) else []

unique_circuits = len({e['metadata']['circuit_hash'] for e in paras})
print(f'Paraphrases generated: {len(paras)}')
print(f'Unique circuits:       {unique_circuits} / 9527')
print(f'Avg per circuit:       {len(paras)/unique_circuits:.1f}' if unique_circuits else '')
print(f'Errors logged:         {len(para_errors)}')
print()
print('Sample paraphrase group:')
first_hash = paras[0]['metadata']['circuit_hash']
group = [e for e in paras if e['metadata']['circuit_hash'] == first_hash]
print(f'  Original: {group[0]["metadata"].get("original_prompt", "n/a")}')
for i, e in enumerate(group):
    print(f'  Para {i+1}:  {e["input"]}')

In [ ]:
# ── Cell 14: Batch 2 Stage 3 — Merge into main dataset ────────────────────
# Merges seeds_expansion + paraphrases_expansion into train_clean / validation_clean
# Safe to re-run: deduplicates by input text before writing

STAGE3_EXP = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
              r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
              r'\scripts\03_instruction_generation\merge_expansion_entries.py')

run_script(STAGE3_EXP)

In [ ]:
# ── Cell 15: Final dataset summary ────────────────────────────────────────
train_final = load_jsonl(f'{BASE}/train_clean.jsonl')
val_final   = load_jsonl(f'{BASE}/validation_clean.jsonl')

total = len(train_final) + len(val_final)
flag_counts = {}
for e in train_final + val_final:
    f = e['metadata'].get('quality_flag', 'unknown')
    flag_counts[f] = flag_counts.get(f, 0) + 1

print('=== FINAL DATASET SUMMARY (after Batch 2) ===')
print(f'Train:  {len(train_final)}')
print(f'Val:    {len(val_final)}')
print(f'Total:  {total}')
print()
print('By quality_flag:')
for f, c in sorted(flag_counts.items()):
    print(f'  {f:<22} {c:>6}  ({100*c/total:.1f}%)')

## Batch 3 — Additional GitHub Queries + Qiskit Official Sources

Scrapes 8 new GitHub search queries and Qiskit official repos (tutorials, textbook, community tutorials).

**Run cells in order:**
- Cell 16 — GitHub + Qiskit scraping → circuits_expansion_v2.jsonl
- Cell 17 — Seed generation
- Cell 18 — Check seeds
- Cell 19 — Paraphrase generation
- Cell 20 — Check paraphrases
- Cell 21 — Merge into main dataset
- Cell 22 — Final dataset summary

In [13]:
# ── Cell 16: Batch 3 — GitHub + Qiskit official scraping ──────────────────
# Searches 8 new GitHub queries + Qiskit tutorials/textbook repos
# Output: circuits_expansion_v2.jsonl
# Resume-safe: re-running skips already-fetched URLs

SCRAPE_V2 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
             r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
             r'\scripts\scrape_github_expansion_v2.py')

run_script(SCRAPE_V2)

ERROR: GITHUB_TOKEN environment variable is not set.
Export a Personal Access Token:  export GITHUB_TOKEN=ghp_...

Script exited with code 1


In [ ]:
# ── Cell 17: Batch 3 Stage 1 — Seed generation ────────────────────────────
SEEDS_V2 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
            r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
            r'\scripts\03_instruction_generation\generate_seeds_expansion_v2.py')

run_script(SEEDS_V2)

In [ ]:
# ── Cell 18: Check Batch 3 Stage 1 output ─────────────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

seeds_v2 = load_jsonl(f'{BASE}/seeds_expansion_v2.jsonl')
errors_v2_path = f'{BASE}/seeds_expansion_v2_errors.jsonl'
errors_v2 = load_jsonl(errors_v2_path) if os.path.exists(errors_v2_path) else []

n_circuits = sum(1 for l in open(f'{BASE}/circuits_expansion_v2.jsonl', encoding='utf-8') if l.strip())
print(f'Seeds generated:  {len(seeds_v2)} / {n_circuits}')
print(f'Errors logged:    {len(errors_v2)}')
print()
for e in seeds_v2[:3]:
    print(f'  INPUT:  {e["input"]}')
    print(f'  OUTPUT: {e["output"][:80]}...')
    print()

In [ ]:
# ── Cell 19: Batch 3 Stage 2 — Paraphrase generation ──────────────────────
PARAS_V2 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
            r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
            r'\scripts\03_instruction_generation\generate_paraphrases_expansion_v2.py')

run_script(PARAS_V2)

In [ ]:
# ── Cell 20: Check Batch 3 Stage 2 output ─────────────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

paras_v2 = load_jsonl(f'{BASE}/paraphrases_expansion_v2.jsonl')
para_v2_errors_path = f'{BASE}/paraphrases_expansion_v2_errors.jsonl'
para_v2_errors = load_jsonl(para_v2_errors_path) if os.path.exists(para_v2_errors_path) else []

unique_v2 = len({e['metadata']['circuit_hash'] for e in paras_v2})
print(f'Paraphrases generated: {len(paras_v2)}')
print(f'Unique circuits:       {unique_v2}')
print(f'Avg per circuit:       {len(paras_v2)/unique_v2:.1f}' if unique_v2 else '')
print(f'Errors logged:         {len(para_v2_errors)}')

In [ ]:
# ── Cell 21: Batch 3 Stage 3 — Merge into main dataset ────────────────────
MERGE_V2 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
            r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
            r'\scripts\03_instruction_generation\merge_expansion_v2_entries.py')

run_script(MERGE_V2)

In [ ]:
# ── Cell 22: Final dataset summary (after Batch 3) ────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_final = load_jsonl(f'{BASE}/train_clean.jsonl')
val_final   = load_jsonl(f'{BASE}/validation_clean.jsonl')

total = len(train_final) + len(val_final)
flag_counts = {}
source_counts = {}
for e in train_final + val_final:
    f = e['metadata'].get('quality_flag', 'unknown')
    flag_counts[f] = flag_counts.get(f, 0) + 1
    s = e['metadata'].get('source_dataset', 'unknown')
    source_counts[s] = source_counts.get(s, 0) + 1

print('=== FINAL DATASET SUMMARY (after Batch 3) ===')
print(f'Train:  {len(train_final)}')
print(f'Val:    {len(val_final)}')
print(f'Total:  {total}')
print()
print('By quality_flag:')
for f, c in sorted(flag_counts.items()):
    print(f'  {f:<22} {c:>6}  ({100*c/total:.1f}%)')
print()
print('By source_dataset:')
for s, c in sorted(source_counts.items()):
    print(f'  {s:<22} {c:>6}  ({100*c/total:.1f}%)')

## Batch 4 — GitHub Topics API + Organization Scraping + New Code Queries

Scrapes GitHub Topics (qiskit, quantum-computing, quantum-circuit, quantum-machine-learning), organizations (qiskit-community, IBM-Quantum, Qiskit), and 8 new code search queries.

Cells 23–29:
- **Cell 23**: Scrape circuits → `circuits_expansion_v3.jsonl`
- **Cell 24**: Generate seed instructions → `seeds_expansion_v3.jsonl`
- **Cell 25**: Check seeds
- **Cell 26**: Generate paraphrases → `paraphrases_expansion_v3.jsonl`
- **Cell 27**: Check paraphrases
- **Cell 28**: Merge into `train_clean.jsonl` / `validation_clean.jsonl`
- **Cell 29**: Dataset summary

In [ ]:
# ── Cell 23: Batch 4 — GitHub Topics + Org scraping ───────────────────────
import os
GITHUB_TOKEN_PATH = r'C:\Users\Abebe\Downloads\IT\GITHUB\GITHUB_TOKEN_PQID_V1.txt'
with open(GITHUB_TOKEN_PATH, 'r') as f:
    os.environ['GITHUB_TOKEN'] = f.read().strip()
print('GitHub token loaded:', 'ghp_...' + os.environ['GITHUB_TOKEN'][-6:])

SCRAPE_V3 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
             r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
             r'\scripts\scrape_github_expansion_v3.py')
run_script(SCRAPE_V3)

In [ ]:
# ── Cell 24: Batch 4 Stage 1 — Seed generation ────────────────────────────
SEEDS_V3 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
            r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
            r'\scripts\03_instruction_generation\generate_seeds_expansion_v3.py')
run_script(SEEDS_V3)

In [ ]:
# ── Cell 25: Check Batch 4 Stage 1 output ─────────────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

seeds_v3 = load_jsonl(f'{BASE}/seeds_expansion_v3.jsonl')
errors_v3_path = f'{BASE}/seeds_expansion_v3_errors.jsonl'
errors_v3 = load_jsonl(errors_v3_path) if os.path.exists(errors_v3_path) else []
n_v3 = sum(1 for l in open(f'{BASE}/circuits_expansion_v3.jsonl', encoding='utf-8') if l.strip())
print(f'Seeds generated:  {len(seeds_v3)} / {n_v3}')
print(f'Errors logged:    {len(errors_v3)}')
for e in seeds_v3[:3]:
    print(f'  INPUT:  {e["input"]}')
    print(f'  OUTPUT: {e["output"][:80]}...')
    print()

In [ ]:
# ── Cell 26: Batch 4 Stage 2 — Paraphrase generation ──────────────────────
PARAS_V3 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
            r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
            r'\scripts\03_instruction_generation\generate_paraphrases_expansion_v3.py')
run_script(PARAS_V3)

In [ ]:
# ── Cell 27: Check Batch 4 Stage 2 output ─────────────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

paras_v3 = load_jsonl(f'{BASE}/paraphrases_expansion_v3.jsonl')
para_v3_errors_path = f'{BASE}/paraphrases_expansion_v3_errors.jsonl'
para_v3_errors = load_jsonl(para_v3_errors_path) if os.path.exists(para_v3_errors_path) else []
unique_v3 = len({e['metadata']['circuit_hash'] for e in paras_v3})
print(f'Paraphrases generated: {len(paras_v3)}')
print(f'Unique circuits:       {unique_v3}')
print(f'Avg per circuit:       {len(paras_v3)/unique_v3:.1f}' if unique_v3 else '')
print(f'Errors logged:         {len(para_v3_errors)}')

In [ ]:
# ── Cell 28: Batch 4 Stage 3 — Merge into main dataset ──────────────────────
MERGE_V3 = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\scripts\03_instruction_generation' + r'\merge_expansion_v3_entries.py')
run_script(MERGE_V3)

In [36]:
# ── Cell 29: Final dataset summary (after Batch 4) ──────────────────────────
import json, os

BASE = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\data\processed'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_b4 = load_jsonl(f'{BASE}/train_clean.jsonl')
val_b4   = load_jsonl(f'{BASE}/validation_clean.jsonl')

total = len(train_b4) + len(val_b4)
flag_counts = {}
for e in train_b4 + val_b4:
    fl = e['metadata'].get('quality_flag', 'unknown')
    flag_counts[fl] = flag_counts.get(fl, 0) + 1

print('=== DATASET SUMMARY (after Batch 4) ===')
print(f'Train:  {len(train_b4)}')
print(f'Val:    {len(val_b4)}')
print(f'Total:  {total}')
print()
print('By quality_flag:')
for fl, c in sorted(flag_counts.items()):
    print(f'  {fl:<22} {c:>6}  ({100*c/total:.1f}%)')


=== DATASET SUMMARY (after Batch 4) ===
Train:  516540
Val:    174511
Total:  691051

By quality_flag:
  clean                    1903  (0.3%)
  new_scraped            682575  (98.8%)
  rescraped                4210  (0.6%)
  rescued                  1118  (0.2%)
  revlib                   1245  (0.2%)


In [37]:
# ── Cell 30: Enrich metadata with GitHub repo topics ────────────────────────
# Calls GitHub API once per unique repo to fetch its topic tags.
# Patches all scrape files + train/val with  repo_topics: [...]
# Safe to re-run (skips entries that already have repo_topics).
# Run AFTER all batch scraping is complete.

ENRICH_TOPICS = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\scripts' + r'\enrich_repo_topics.py')
run_script(ENRICH_TOPICS)

GitHub token loaded: gith...k5jN
Scanning files for unique repos ...
  circuits_expansion.jsonl: 9527 entries
  circuits_expansion_v2.jsonl: 7810 entries
  circuits_expansion_v3.jsonl: 94321 entries
  train_clean.jsonl: 516540 entries
  validation_clean.jsonl: 174511 entries
  test_clean.jsonl: 0 entries

Unique GitHub repos to look up: 4640

Fetching repo info from GitHub API ...
  [1/4640] 00PrabalK00/ContractEncrypt [user] topics=[]
  [2/4640] 0Ishtar0/QASC [user] topics=[]
  [3/4640] 0penAGI/0p3q [user] topics=[]
  [4/4640] 0xBoji/quantum-computing-examples [user] topics=['grover-algorithm', 'qiskit', 'quantum-algorithms', 'quantum-computing', 'quantum-teleportation']
  [5/4640] 0xC4LL3/AVQE [user] topics=[]
  [6/4640] 0xSooki/qcbm [user] topics=['quantum-computing', 'quantum-machine-learning']
  [7/4640] 10-05-15/Q-Network-Simulator [user] topics=[]
  [8/4640] 10ay/Qiskit-quantum-teleportation [user] topics=[]
  [9/4640] 10srav/QKD-Simulator-Project [user] topics=[]
  [10/4640] 10

## Final Step — Test Split + Metadata Enrichment + Validation

Run after all batches (1–4) are merged into train_clean.jsonl / validation_clean.jsonl.

Cells 30–34:
- **Cell 30**: Enrich all entries with GitHub repo topics (`enrich_repo_topics.py`)
- **Cell 31**: Create 80/10/10 train/val/test split → `test_clean.jsonl`
- **Cell 32**: Backfill XAI metadata fields (`patch_metadata.py`)
- **Cell 33**: Final metadata enrichment — circuit stats + validation_status (`enrich_metadata.py`)
- **Cell 34**: Partition into Tier 1 (validated) + Tier 2 (community) (`split_validated.py`)

⚠️  `test_clean.jsonl` / `test_validated.jsonl` must NEVER be loaded during training.

In [38]:
# ── Cell 31: Create test split ─────────────────────────────────────────────
# Carves out a permanent 80/10/10 train/val/test split at circuit level.
# Stratified by quality_flag. Seeds go to test, paraphrases to train/val.
# Output: train_clean.jsonl (80%), validation_clean.jsonl (10%), test_clean.jsonl (10%)
# ⚠️  test_clean.jsonl must NEVER be loaded during training or fine-tuning.

SPLIT_TEST = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
              r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
              r'\scripts\split_test.py')

run_script(SPLIT_TEST)

Loading data â€¦
  Total entries loaded: 691,051
  Unique circuits: 115,471

Circuit assignments per quality_flag:
  flag             total   train     val    test
  clean              265     213      26      26
  new_scraped    113,955  91,163  11,396  11,396
  rescraped          835     667      84      84
  rescued            167     133      17      17
  revlib             249     199      25      25

Writing temp files â€¦
  Wrote 604,666 entries â†’ c:/Users/Abebe/Downloads/CAREER/ACADEMIC CAREER/SCHOOLS/YONSEI/YONSEI 2023/Yonsei SS 2025/MS Thesis/MS_THESIS_DATASET/PQID/data/processed/train_split_tmp.jsonl
  Wrote 74,837 entries â†’ c:/Users/Abebe/Downloads/CAREER/ACADEMIC CAREER/SCHOOLS/YONSEI/YONSEI 2023/Yonsei SS 2025/MS Thesis/MS_THESIS_DATASET/PQID/data/processed/validation_split_tmp.jsonl
  Wrote 11,548 entries â†’ c:/Users/Abebe/Downloads/CAREER/ACADEMIC CAREER/SCHOOLS/YONSEI/YONSEI 2023/Yonsei SS 2025/MS Thesis/MS_THESIS_DATASET/PQID/data/processed/test_split_tmp.jsonl



In [39]:
# ── Cell 32: Backfill XAI metadata fields ─────────────────────────────────
# Adds generation_model, generation_date, prompt_type, paraphrase_source
# to all entries that are missing them. Safe to re-run (never overwrites).
# Run AFTER test split (Cell 23) so test_clean.jsonl is also patched.

PATCH = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
         r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
         r'\scripts\patch_metadata.py')

run_script(PATCH)

patch_metadata.py â€” XAI metadata backfill
Base directory : c:/Users/Abebe/Downloads/CAREER/ACADEMIC CAREER/SCHOOLS/YONSEI/YONSEI 2023/Yonsei SS 2025/MS Thesis/MS_THESIS_DATASET/PQID/data/processed

Processing: train_clean.jsonl ...
  Records processed : 604666
  Fields backfilled :
    generation_model       604,666 records updated
    generation_date        604,666 records updated
    prompt_type                  0 records updated
    paraphrase_source      514,193 records updated

Processing: validation_clean.jsonl ...
  Records processed : 74837
  Fields backfilled :
    generation_model        74,837 records updated
    generation_date         74,837 records updated
    prompt_type                  0 records updated
    paraphrase_source       63,507 records updated

Processing: test_clean.jsonl ...
  Records processed : 11548
  Fields backfilled :
    generation_model        11,548 records updated
    generation_date         11,548 records updated
    prompt_type                

In [ ]:
# ── Cell 33: Final metadata enrichment (post all batches) ──────────────────
# Re-runs enrichment on the complete final dataset after test split.
# Updates train_clean.jsonl, validation_clean.jsonl, and test_clean.jsonl.

import os

# Patch enrich_metadata.py to also process test_clean.jsonl
ENRICH = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI'
          r'\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID'
          r'\scripts\enrich_metadata.py')

run_script(ENRICH)

Importing Qiskit...


Qiskit ready.

[train] Loading c:/Users/Abebe/Downloads/CAREER/ACADEMIC CAREER/SCHOOLS/YONSEI/YONSEI 2023/Yonsei SS 2025/MS Thesis/MS_THESIS_DATASET/PQID/data/processed/train_clean.jsonl ...
[train] 604666 entries to process.
  [train] 500/604666 (0.1%)  enriched=292  skipped=208  elapsed=35s
Missing variables. Make sure `scores` and `labels` exist from your test step.Missing variables. Make sure `scores` and `labels` exist from your test step.
Error: name 'scores' is not defined
Missing variables. Make sure `scores` and `labels` exist from your test step.
Error: name 'scores' is not defined

Error: name 'scores' is not defined
Missing variables. Make sure `scores` and `labels` exist from your test step.
Error: name 'scores' is not defined
Missing variables. Make sure `scores` and `labels` exist from your test step.
Error: name 'scores' is not defined
Missing variables. Make sure `scores` and `labels` exist from your test step.
Error: name 'scores' is not defined
  [train] 1000/604666 

In [ ]:
# ── Cell 35: Circuit family & semantic intent classification ──────
# Uses gpt-4.1-mini to label each unique circuit_hash with:
#   circuit_family  (bell | ghz | qft | variational | qaoa | ...)
#   semantic_intent (state_preparation | entanglement_generation | ...)
# Resume-safe: cached in circuit_family_cache.jsonl.
# Run AFTER Cell 33 (enrich_metadata) and BEFORE Cell 36 (split_validated).
import os
with open(r'C:\Users\Abebe\Downloads\IT\OPENAI\OPENAI_API_KEY_PQID_V2.txt') as _f:
    os.environ['OPENAI_API_KEY'] = _f.read().strip()
FAMILY_SCRIPT = r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\scripts\enrich_circuit_family.py'
run_script(FAMILY_SCRIPT)

In [ ]:
# ── Cell 34: Split into validated / community-unvalidated tiers ─────────────
# Tier 1 (validated): circuit executed cleanly in Qiskit -> benchmark dataset
# Tier 2 (community): everything else, labelled by failure reason -> public curation
# Outputs: train/validation/test_validated.jsonl + community_unvalidated.jsonl
# Run AFTER Cell 33 (final metadata enrichment).

SPLIT_VAL = (r'C:\Users\Abebe\Downloads\CAREER\ACADEMIC CAREER\SCHOOLS\YONSEI\YONSEI 2023\Yonsei SS 2025\MS Thesis\MS_THESIS_DATASET\PQID\scripts' + r'\split_validated.py')
run_script(SPLIT_VAL)